# ZeroDefect — YOLOv11n Infrastructure Defect Detection
**Train a custom YOLOv11n model to detect: crack, spalling, exposed rebar, rust, scaling, efflorescence**

⚡ Make sure GPU is enabled: Runtime → Change runtime type → T4 GPU

Training takes ~1-2 hours on Colab T4.

## Step 1 — Install dependencies

In [ ]:
!pip install ultralytics roboflow -q
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## Step 2 — Download dataset from Roboflow

We're using the **Concrete Defect Detection** dataset by SHM:
- 1,680 images
- 6 classes: `crack`, `Spalling`, `Exposed_reinforcement`, `Ruststrain`, `Scaling`, `efflorescence`
- Already annotated in YOLO format

You'll need a free Roboflow account. When you run the cell below, it will ask you to authenticate.

In [ ]:
from roboflow import Roboflow

# This will open a browser tab to authenticate — sign up for free if needed
rf = Roboflow()

project = rf.workspace("shm-agftj").project("concrete-defect-detection-pl8ed")
version = project.version(2)
dataset = version.download("yolov11")

print(f"\n✅ Dataset downloaded to: {dataset.location}")
print(f"Classes: {project.classes}")

## Step 2b — Also merge a larger crack-focused dataset
Since you want to focus heavily on cracks, let's add more crack images from the Concrete Defects dataset (6,913 images).

In [ ]:
# Download the larger SHM Concrete-defects dataset for more crack samples
project2 = rf.workspace("shm-agftj").project("concrete-defects-oijvr")
version2 = project2.version(1)
dataset2 = version2.download("yolov11")

print(f"\n✅ Extra dataset downloaded to: {dataset2.location}")

## Step 2c — Merge datasets into one training folder

In [ ]:
import shutil
import os
import glob
import yaml

# Create merged dataset directory
merged_dir = "/content/merged_dataset"
for split in ["train", "valid", "test"]:
    os.makedirs(f"{merged_dir}/{split}/images", exist_ok=True)
    os.makedirs(f"{merged_dir}/{split}/labels", exist_ok=True)

# Copy from primary dataset
for split in ["train", "valid", "test"]:
    src_img = f"{dataset.location}/{split}/images"
    src_lbl = f"{dataset.location}/{split}/labels"
    if os.path.exists(src_img):
        for f in glob.glob(f"{src_img}/*"):
            shutil.copy2(f, f"{merged_dir}/{split}/images/")
    if os.path.exists(src_lbl):
        for f in glob.glob(f"{src_lbl}/*"):
            shutil.copy2(f, f"{merged_dir}/{split}/labels/")

# Copy from secondary dataset (prefix filenames to avoid collisions)
for split in ["train", "valid", "test"]:
    src_img = f"{dataset2.location}/{split}/images"
    src_lbl = f"{dataset2.location}/{split}/labels"
    if os.path.exists(src_img):
        for f in glob.glob(f"{src_img}/*"):
            name = os.path.basename(f)
            shutil.copy2(f, f"{merged_dir}/{split}/images/ds2_{name}")
    if os.path.exists(src_lbl):
        for f in glob.glob(f"{src_lbl}/*"):
            name = os.path.basename(f)
            shutil.copy2(f, f"{merged_dir}/{split}/labels/ds2_{name}")

# Count merged images
for split in ["train", "valid", "test"]:
    count = len(glob.glob(f"{merged_dir}/{split}/images/*"))
    print(f"{split}: {count} images")

## Step 2d — Create unified data.yaml

In [ ]:
# Create the unified YAML config
# Standardized class names for ZeroDefect
data_yaml = {
    "path": merged_dir,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {
        0: "crack",
        1: "spalling",
        2: "exposed_rebar",
        3: "rust",
        4: "scaling",
        5: "efflorescence"
    },
    "nc": 6
}

yaml_path = f"{merged_dir}/data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"✅ data.yaml created at {yaml_path}")
print(f"Classes: {data_yaml['names']}")

## Step 3 — Verify dataset
Quick sanity check — view a few training images with labels.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

train_imgs = sorted(glob.glob(f"{merged_dir}/train/images/*"))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, train_imgs):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Try to load corresponding label
    lbl_path = img_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    title = os.path.basename(img_path)
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            n_boxes = len(f.readlines())
        title += f" ({n_boxes} boxes)"
    
    ax.imshow(img)
    ax.set_title(title, fontsize=8)
    ax.axis("off")

plt.suptitle("Sample Training Images", fontsize=14)
plt.tight_layout()
plt.show()

## Step 4 — Train YOLOv11n

Training config:
- **Model:** YOLOv11n (nano — fastest, best for Jetson)
- **Epochs:** 100 (with early stopping patience=15)
- **Image size:** 640px
- **Batch:** 16 (fits T4 GPU)
- **Augmentation:** Mosaic, flip, HSV jitter (built into Ultralytics)

⏱️ Takes ~1-2 hours on T4. Watch the mAP50 metric — aim for >0.5

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv11n
model = YOLO("yolo11n.pt")

# Train
results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=15,           # early stopping — stops if no improvement for 15 epochs
    device=0,              # GPU
    workers=2,
    project="zerodefect",
    name="yolo11n_defects",
    
    # Augmentation settings
    mosaic=1.0,
    flipud=0.5,            # vertical flip (drone views can be any orientation)
    fliplr=0.5,            # horizontal flip
    hsv_h=0.015,           # hue augmentation
    hsv_s=0.7,             # saturation augmentation
    hsv_v=0.4,             # value/brightness augmentation
    scale=0.5,             # scale augmentation (simulates different altitudes)
    
    # Save best model
    save=True,
    save_period=10,        # checkpoint every 10 epochs
    plots=True,
    verbose=True,
)

print("\n✅ Training complete!")

## Step 5 — Evaluate results

In [ ]:
# Show training curves
from IPython.display import Image, display

# Results plot
results_img = "zerodefect/yolo11n_defects/results.png"
if os.path.exists(results_img):
    display(Image(filename=results_img, width=900))

# Confusion matrix
cm_img = "zerodefect/yolo11n_defects/confusion_matrix.png"
if os.path.exists(cm_img):
    display(Image(filename=cm_img, width=600))

In [ ]:
# Validate on test set
best_model = YOLO("zerodefect/yolo11n_defects/weights/best.pt")
metrics = best_model.val(data=yaml_path, split="test")

print(f"\n{'='*50}")
print(f"TEST SET RESULTS")
print(f"{'='*50}")
print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")
print(f"{'='*50}")

## Step 6 — Test on sample images

In [ ]:
# Run inference on test images
test_imgs = sorted(glob.glob(f"{merged_dir}/test/images/*"))[:8]

results = best_model.predict(
    source=test_imgs,
    conf=0.35,
    iou=0.45,
    save=True,
    project="zerodefect",
    name="test_predictions",
)

# Display predictions
pred_imgs = sorted(glob.glob("zerodefect/test_predictions/*"))[:8]
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, img_path in zip(axes.flat, pred_imgs):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.axis("off")

plt.suptitle("YOLOv11n Predictions on Test Images", fontsize=14)
plt.tight_layout()
plt.show()

## Step 7 — Export for deployment

Export the model in multiple formats:
- `.pt` — for your ground station backend
- `ONNX` — for Jetson (convert to TensorRT on device)
- `TensorRT` — if Colab has the right CUDA version

In [ ]:
# Export to ONNX (works everywhere, convert to TensorRT on Jetson)
best_model.export(format="onnx", imgsz=640, simplify=True)
print("\n✅ ONNX export complete")

# Try TensorRT export (may not work on all Colab instances)
try:
    best_model.export(format="engine", imgsz=640, half=True)  # FP16 for Jetson
    print("✅ TensorRT export complete")
except Exception as e:
    print(f"⚠️ TensorRT export failed (expected on some Colab instances): {e}")
    print("   → Export ONNX to Jetson and convert there with:")
    print('   → yolo export model=best.pt format=engine device=0 half=True')

## Step 8 — Download your trained model

Download these files to your machine:
- `best.pt` → goes in `hawki/models/` for ground station
- `best.onnx` → transfer to Jetson for TensorRT conversion

In [ ]:
from google.colab import files
import shutil

# Copy best weights to easy download location
weights_dir = "zerodefect/yolo11n_defects/weights"

# Download best.pt
print("Downloading best.pt (PyTorch weights)...")
files.download(f"{weights_dir}/best.pt")

# Download ONNX if it exists
onnx_path = f"{weights_dir}/best.onnx"
if os.path.exists(onnx_path):
    print("Downloading best.onnx (ONNX for Jetson)...")
    files.download(onnx_path)

print("\n✅ Download complete!")
print("\nNext steps:")
print("1. Put best.pt in hawki/models/zerodefect_yolo11n.pt")
print("2. Transfer best.onnx to Jetson")
print("3. On Jetson run: yolo export model=best.onnx format=engine half=True")

## Quick test — Load and run your trained model

In [ ]:
# Quick inference test to make sure model works
model = YOLO(f"{weights_dir}/best.pt")

# Pick a random test image
test_img = sorted(glob.glob(f"{merged_dir}/test/images/*"))[0]
results = model.predict(test_img, conf=0.35, verbose=False)

for r in results:
    for box in r.boxes:
        cls_name = model.names[int(box.cls)]
        conf = float(box.conf)
        coords = box.xyxy[0].tolist()
        print(f"  Detected: {cls_name} | conf: {conf:.2f} | box: {[round(c) for c in coords]}")

print(f"\n✅ Model working! {len(results[0].boxes)} detections found.")